# 🎭 PoetryDuel-GPT — Colab Training

**Two-stage training with rhyme/tone conditioning (~30M params).**

| Step | Time |
|------|------|
| Clone + install | ~2 min |
| Preprocess + tokenize | ~3 min |
| Stage 1: All genres (15K steps) | ~3 hr T4 / ~2 hr L4 |
| Stage 2: Lục Bát fine-tune (5K steps) | ~1 hr T4 / ~40 min L4 |
| Generate poetry | ~1 min |

Data is already in the repo (`poems_dataset_clean.csv`). No Drive mount needed for data.
Mount Drive in Cell 6 only to save checkpoints.

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 2. Preprocess + Tokenize (~3 min)
# poems_dataset_clean.csv is already in the repo — skip cleaning
%cd /content/poetry-dual-gpt

# Generate training pairs (Lục Bát + Thất Ngôn with rhyme/tone tags)
!python src/preprocess.py

# Train BPE tokenizer
!python src/train_bpe.py

In [ ]:
# @title 3. Stage 1 — Pretrain on ALL genres (~3 hr T4, ~2 hr L4)
# Resume-safe: re-run if Colab disconnects
assert torch.cuda.is_available(), "⚠️  Enable GPU: Runtime → Change runtime type → T4 GPU"

%cd /content/poetry-dual-gpt
import glob
latest = sorted(glob.glob('checkpoints/stage1_step_*.pt'))
if latest:
    print(f'📂 Resuming from {latest[-1]}')
    !python src/train.py --mode train --resume {latest[-1]}
else:
    !python src/train.py --mode train

# Rename checkpoints for Stage 1
!cp checkpoints/final.pt checkpoints/stage1_final.pt
!cp checkpoints/best.pt checkpoints/stage1_best.pt 2>/dev/null || true
for f in sorted(glob.glob('checkpoints/step_*.pt')):
    new = f.replace('checkpoints/step_', 'checkpoints/stage1_step_')
    !mv {f} {new}
print('✅ Stage 1 → checkpoints/stage1_final.pt')

In [ ]:
# @title 4. Prepare Lục Bát-only corpus (~30 sec)
%cd /content/poetry-dual-gpt

with open('data/poetry_corpus.txt') as f:
    lines = f.readlines()
luc_bat = [l for l in lines if '[LUC_BAT]' in l]
with open('data/corpus_luc_bat.txt', 'w') as f:
    f.writelines(luc_bat)
print(f'Lục Bát pairs: {len(luc_bat):,} / {len(lines):,} total')

In [ ]:
# @title 5. Stage 2 — Fine-tune on Lục Bát only (~1 hr T4, ~40 min L4)
# Resume-safe: re-run if interrupted
assert torch.cuda.is_available(), "⚠️  Enable GPU"

%cd /content/poetry-dual-gpt
import glob
latest = sorted(glob.glob('checkpoints/stage2_step_*.pt'))
if latest:
    print(f'📂 Resuming from {latest[-1]}')
    !python src/train.py --resume {latest[-1]} --corpus data/corpus_luc_bat.txt --steps 5000
else:
    !python src/train.py \
        --resume checkpoints/stage1_final.pt \
        --corpus data/corpus_luc_bat.txt \
        --steps 5000

!cp checkpoints/final.pt checkpoints/stage2_final.pt
!cp checkpoints/best.pt checkpoints/stage2_best.pt 2>/dev/null || true
for f in sorted(glob.glob('checkpoints/step_*.pt')):
    new = f.replace('checkpoints/step_', 'checkpoints/stage2_step_')
    !mv {f} {new}
print()
print('🎉 Two-stage training complete!')
print('   Stage 1: checkpoints/stage1_final.pt')
print('   Stage 2: checkpoints/stage2_final.pt')

In [ ]:
# @title 6. Generate Poetry
%cd /content/poetry-dual-gpt

!python src/sample.py \
    --checkpoint checkpoints/stage2_final.pt \
    --temperature 0.75 \
    --top_p 0.92 \
    --num_samples 3

# Compare Stage 1 vs Stage 2:
# !python src/sample.py --checkpoint checkpoints/stage1_final.pt --num_samples 2

# Interactive (run in separate cell):
# !python src/sample.py --checkpoint checkpoints/stage2_final.pt --interactive

In [ ]:
# @title 7. Save to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = "/content/drive/MyDrive/poetry-dual-gpt/checkpoints"
!mkdir -p {DRIVE_CKPT}

import glob
for ckpt in sorted(glob.glob('checkpoints/stage*_*.pt')):
    !cp -v {ckpt} {DRIVE_CKPT}/
print('✅ Checkpoints saved to Drive')